# 从零构建AI推理大模型 - 一位数加减乘除推理

## 项目概述
- **目标**: 从零训练一个小型语言模型，使其具备一位数加减乘除的推理能力
- **训练策略**: SFT（监督微调）+ GRPO（群组相对策略优化）
- **模型架构**: GPT风格 Decoder-Only Transformer（RoPE + RMSNorm + SwiGLU）
- **平台**: Kaggle T4 GPU (16GB VRAM)
- **参数量**: ~15M

## 训练流程
1. 生成带推理过程的数据集
2. 构建自定义Tokenizer
3. 定义模型架构
4. SFT监督微调
5. GRPO强化学习
6. 评估与推理演示

## Step 0: 环境配置

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math
import random
import os
import time
import copy
from collections import Counter
from typing import List, Dict, Tuple, Optional

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
class Config:
    vocab_size = 200
    max_seq_len = 128
    d_model = 512
    n_heads = 8
    n_layers = 6
    d_ff = 1408
    dropout = 0.1
    batch_size = 32
    sft_lr = 3e-4
    weight_decay = 0.1
    warmup_steps = 200
    sft_epochs = 40
    grad_clip = 1.0
    grpo_group_size = 4
    grpo_clip_ratio = 0.2
    grpo_kl_coef = 0.05
    grpo_steps = 300
    grpo_lr = 3e-5
    grpo_batch_size = 16
    seed = 42

config = Config()
torch.manual_seed(config.seed)
random.seed(config.seed)
print('Config loaded.')

## Step 1: 数据集生成

生成一位数加减乘除的完整数据集，包含多种中文推理模板：
- 加法: 0-9 + 0-9
- 减法: 0-9 - 0-9（允许负数结果）
- 乘法: 0-9 x 0-9
- 除法: 0-9 / 1-9（带余数）

In [ ]:
class ArithmeticDataGenerator:
    def __init__(self, seed=42):
        random.seed(seed)

    def generate(self):
        data = []
        data.extend(self._gen_addition())
        data.extend(self._gen_subtraction())
        data.extend(self._gen_multiplication())
        data.extend(self._gen_division())
        random.shuffle(data)
        return data

    def _gen_addition(self):
        results = []
        for a in range(10):
            for b in range(10):
                s = a + b
                templates = [
                    f"计算{a}加{b}。{a}与{b}相加，{a}+{b}={s}，所以答案是{s}。",
                    f"求{a}+{b}的值。{a}加上{b}等于{s}，因此结果是{s}。",
                    f"{a}加{b}等于多少？{a}和{b}做加法运算，{a}+{b}={s}，答案为{s}。",
                    f"算一算{a}+{b}。从{a}开始往后数{b}个数，得到{s}，所以{a}+{b}={s}。",
                    f"计算{a}+{b}。{a}加上{b}，和为{s}。",
                    f"{a}+{b}怎么算？{a}加{b}等于{s}，答案是{s}。",
                    f"把{a}和{b}加起来。{a}+{b}={s}，结果是{s}。",
                    f"{a}加上{b}得多少？{a}加{b}等于{s}，所以得{s}。",
                ]
                for t in templates:
                    results.append({
                        'prompt': f'计算: {a} + {b} = ?',
                        'reasoning': t,
                        'answer': str(s),
                        'operation': '+',
                        'full_text': f'计算: {a} + {b} = ?\n推理: {t}\n答案: {s}'
                    })
        return results

    def _gen_subtraction(self):
        results = []
        for a in range(10):
            for b in range(10):
                d = a - b
                templates = [
                    f"计算{a}减{b}。{a}与{b}相减，{a}-{b}={d}，所以答案是{d}。",
                    f"求{a}-{b}的值。{a}减去{b}等于{d}，因此结果是{d}。",
                    f"{a}减{b}等于多少？{a}和{b}做减法运算，{a}-{b}={d}，答案为{d}。",
                    f"算一算{a}-{b}。从{a}开始往回数{b}个数，得到{d}，所以{a}-{b}={d}。",
                    f"计算{a}-{b}。{a}减去{b}，差为{d}。",
                    f"{a}-{b}怎么算？{a}减{b}等于{d}，答案是{d}。",
                    f"从{a}中去掉{b}。{a}-{b}={d}，结果是{d}。",
                    f"{a}减去{b}得多少？{a}减{b}等于{d}，所以得{d}。",
                ]
                for t in templates:
                    results.append({
                        'prompt': f'计算: {a} - {b} = ?',
                        'reasoning': t,
                        'answer': str(d),
                        'operation': '-',
                        'full_text': f'计算: {a} - {b} = ?\n推理: {t}\n答案: {d}'
                    })
        return results

    def _gen_multiplication(self):
        results = []
        for a in range(10):
            for b in range(10):
                p = a * b
                templates = [
                    f"计算{a}乘{b}。{a}与{b}相乘，{a}×{b}={p}，所以答案是{p}。",
                    f"求{a}×{b}的值。{a}乘以{b}等于{p}，因此结果是{p}。",
                    f"{a}乘{b}等于多少？{a}和{b}做乘法运算，{a}×{b}={p}，答案为{p}。",
                    f"算一算{a}×{b}。{a}的{b}倍是{p}，所以{a}×{b}={p}。",
                    f"计算{a}×{b}。{a}乘以{b}，积为{p}。",
                    f"{a}×{b}怎么算？{a}乘{b}等于{p}，答案是{p}。",
                    f"{a}个{b}相加。{a}×{b}={p}，结果是{p}。",
                    f"{a}的{b}倍是多少？{a}乘以{b}等于{p}，所以是{p}。",
                ]
                for t in templates:
                    results.append({
                        'prompt': f'计算: {a} × {b} = ?',
                        'reasoning': t,
                        'answer': str(p),
                        'operation': '×',
                        'full_text': f'计算: {a} × {b} = ?\n推理: {t}\n答案: {p}'
                    })
        return results

    def _gen_division(self):
        results = []
        for a in range(10):
            for b in range(1, 10):
                q = a // b
                r = a % b
                if r == 0:
                    templates = [
                        f"计算{a}除以{b}。{a}÷{b}={q}，正好整除，所以答案是{q}。",
                        f"求{a}÷{b}的值。{a}除以{b}等于{q}，整除，因此结果是{q}。",
                        f"{a}除以{b}等于多少？{a}和{b}做除法运算，{a}÷{b}={q}，答案为{q}。",
                        f"算一算{a}÷{b}。{a}里面正好包含{q}个{b}，所以{a}÷{b}={q}。",
                        f"计算{a}÷{b}。{a}除以{b}，商为{q}，整除无余数。",
                        f"{a}÷{b}怎么算？{a}除以{b}等于{q}，答案是{q}。",
                        f"{a}能被{b}整除吗？是的，{a}÷{b}={q}，结果是{q}。",
                        f"把{a}平均分成{b}份。{a}÷{b}={q}，每份是{q}。",
                    ]
                    answer = str(q)
                else:
                    templates = [
                        f"计算{a}除以{b}。{a}÷{b}={q}余{r}，所以商是{q}余{r}。",
                        f"求{a}÷{b}的值。{a}除以{b}等于{q}余{r}，因此商为{q}余{r}。",
                        f"{a}除以{b}等于多少？{a}和{b}做除法运算，{a}÷{b}={q}余{r}，答案为商{q}余{r}。",
                        f"算一算{a}÷{b}。{a}里面包含{q}个{b}，还剩{r}，所以{a}÷{b}={q}余{r}。",
                        f"计算{a}÷{b}。{a}除以{b}，商为{q}，余数为{r}。",
                        f"{a}÷{b}怎么算？{a}除以{b}等于{q}余{r}，答案是商{q}余{r}。",
                        f"{a}不能被{b}整除。{a}÷{b}={q}余{r}，结果是商{q}余{r}。",
                        f"把{a}分成每份{b}个。可以分{q}份，还剩{r}个，{a}÷{b}={q}余{r}。",
                    ]
                    answer = f'{q}余{r}'
                for t in templates:
                    results.append({
                        'prompt': f'计算: {a} ÷ {b} = ?',
                        'reasoning': t,
                        'answer': answer,
                        'operation': '÷',
                        'full_text': f'计算: {a} ÷ {b} = ?\n推理: {t}\n答案: {answer}'
                    })
        return results

In [ ]:
generator = ArithmeticDataGenerator(seed=config.seed)
all_data = generator.generate()

print(f'总数据量: {len(all_data)}')
op_counts = Counter(d['operation'] for d in all_data)
for op, cnt in sorted(op_counts.items()):
    print(f'  {op}: {cnt}条')

print(f'\n示例数据:')
for i in [0, 100, 200, 300]:
    if i < len(all_data):
        print(f'---\n{all_data[i]["full_text"]}')

random.shuffle(all_data)
n = len(all_data)
train_data = all_data[:int(n * 0.8)]
val_data = all_data[int(n * 0.8):int(n * 0.9)]
test_data = all_data[int(n * 0.9):]
print(f'\n训练集: {len(train_data)}, 验证集: {len(val_data)}, 测试集: {len(test_data)}')

## Step 2: 构建Tokenizer

基于生成的数据构建自定义字符级Tokenizer，词表包含：
- 数字 0-9
- 运算符 + - × ÷ =
- 中文推理用字
- 特殊Token: <pad>, <bos>, <eos>, <unk>

In [ ]:
class CharTokenizer:
    def __init__(self):
        self.pad_token = '<pad>'
        self.bos_token = '<bos>'
        self.eos_token = '<eos>'
        self.unk_token = '<unk>'
        self.special_tokens = [self.pad_token, self.bos_token, self.eos_token, self.unk_token]
        self.char_to_id = {}
        self.id_to_char = {}
        self.pad_id = 0
        self.bos_id = 1
        self.eos_id = 2
        self.unk_id = 3

    def build_from_data(self, data_list):
        char_set = set()
        for item in data_list:
            for ch in item['full_text']:
                char_set.add(ch)
        sorted_chars = sorted(char_set)
        vocab = self.special_tokens + sorted_chars
        self.char_to_id = {ch: i for i, ch in enumerate(vocab)}
        self.id_to_char = {i: ch for i, ch in enumerate(vocab)}
        self.vocab_size = len(vocab)
        print(f'词表大小: {self.vocab_size}')
        print(f'词表内容: {" ".join(vocab[:50])}...')
        return self

    def encode(self, text):
        return [self.char_to_id.get(ch, self.unk_id) for ch in text]

    def decode(self, ids):
        return ''.join(self.id_to_char.get(i, self.unk_token) for i in ids)

    def encode_with_special(self, text):
        return [self.bos_id] + self.encode(text) + [self.eos_id]

    def decode_without_special(self, ids):
        tokens = []
        for i in ids:
            if i == self.bos_id or i == self.pad_id:
                continue
            if i == self.eos_id:
                break
            tokens.append(self.id_to_char.get(i, self.unk_token))
        return ''.join(tokens)

In [ ]:
tokenizer = CharTokenizer()
tokenizer.build_from_data(all_data)
config.vocab_size = tokenizer.vocab_size

test_text = '计算: 3 + 5 = ?\n推理: 3加5等于8\n答案: 8'
encoded = tokenizer.encode_with_special(test_text)
decoded = tokenizer.decode_without_special(encoded)
print(f'\n编码测试:')
print(f'原文: {repr(test_text)}')
print(f'编码: {encoded}')
print(f'解码: {repr(decoded)}')
print(f'编解码一致: {test_text == decoded}')

## Step 3: 模型架构

实现现代小型GPT模型，采用LLaMA风格的改进：
- **RoPE**: 旋转位置编码
- **RMSNorm**: 替代LayerNorm，更高效
- **SwiGLU**: 替代GELU，效果更好
- **权重共享**: Embedding与输出层共享权重

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=512, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos_cached', emb.cos().unsqueeze(0).unsqueeze(0))
        self.register_buffer('sin_cached', emb.sin().unsqueeze(0).unsqueeze(0))

    def forward(self, seq_len):
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rotary_emb(q, k, cos, sin):
    q = q * cos + rotate_half(q) * sin
    k = k * cos + rotate_half(k) * sin
    return q, k


class Attention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len=512):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        assert d_model % n_heads == 0
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)
        self.attn_dropout = nn.Dropout(0.1)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        cos, sin = self.rotary(T)
        q, k = apply_rotary_emb(q, k, cos, sin)

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)

        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.wo(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len=512):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = Attention(d_model, n_heads, max_seq_len)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.attn_norm(x), mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.d_ff, config.max_seq_len)
            for _ in range(config.n_layers)
        ])
        self.norm = RMSNorm(config.d_model)
        self.head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.token_emb.weight = self.head.weight
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x, targets=None):
        B, T = x.shape
        assert T <= self.config.max_seq_len

        x = self.dropout(self.token_emb(x))
        mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)

        for layer in self.layers:
            x = layer(x, mask)

        x = self.norm(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=0
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100, temperature=0.8, top_k=None, eos_id=2):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
            if next_token.item() == eos_id:
                break
        return idx

In [ ]:
model = GPTModel(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'模型参数量: {total_params:,} ({total_params/1e6:.2f}M)')
print(f'可训练参数量: {trainable_params:,}')

dummy_input = torch.randint(0, config.vocab_size, (2, 32)).to(device)
logits, loss = model(dummy_input, targets=dummy_input)
print(f'测试前向传播: logits shape = {logits.shape}, loss = {loss.item():.4f}')

## Step 4: SFT 监督微调

使用标准的下一个Token预测（Next Token Prediction）进行监督微调，让模型学习推理过程和答案生成。

In [ ]:
class SFTDataset(Dataset):
    def __init__(self, data, tokenizer, max_seq_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.samples = []
        for item in data:
            ids = tokenizer.encode_with_special(item['full_text'])
            if len(ids) <= max_seq_len:
                self.samples.append(ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y


def collate_fn(batch, pad_id=0):
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    xs_padded = []
    ys_padded = []
    for x, y in zip(xs, ys):
        pad_len = max_len - x.size(0)
        xs_padded.append(F.pad(x, (0, pad_len), value=pad_id))
        ys_padded.append(F.pad(y, (0, pad_len), value=pad_id))
    return torch.stack(xs_padded), torch.stack(ys_padded)

In [ ]:
train_dataset = SFTDataset(train_data, tokenizer, config.max_seq_len)
val_dataset = SFTDataset(val_data, tokenizer, config.max_seq_len)
print(f'SFT训练样本数: {len(train_dataset)}')
print(f'SFT验证样本数: {len(val_dataset)}')

train_loader = DataLoader(
    train_dataset, batch_size=config.batch_size, shuffle=True,
    collate_fn=collate_fn, num_workers=0, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=config.batch_size, shuffle=False,
    collate_fn=collate_fn, num_workers=0
)
print(f'训练批次数: {len(train_loader)}')

In [ ]:
class CosineScheduler:
    def __init__(self, optimizer, warmup_steps, max_steps, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        self.min_lr = min_lr
        self.base_lr = optimizer.param_groups[0]['lr']
        self.step_count = 0

    def step(self):
        self.step_count += 1
        if self.step_count < self.warmup_steps:
            lr = self.base_lr * self.step_count / self.warmup_steps
        else:
            progress = (self.step_count - self.warmup_steps) / max(1, self.max_steps - self.warmup_steps)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr

    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']


def train_sft(model, train_loader, val_loader, config, device):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.sft_lr,
        weight_decay=config.weight_decay, betas=(0.9, 0.95)
    )
    max_steps = len(train_loader) * config.sft_epochs
    scheduler = CosineScheduler(optimizer, config.warmup_steps, max_steps)

    best_val_loss = float('inf')
    train_losses = []
    val_losses = []

    for epoch in range(config.sft_epochs):
        model.train()
        epoch_loss = 0
        n_batches = 0
        start_time = time.time()

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            logits, loss = model(batch_x, targets=batch_y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_train_loss = epoch_loss / n_batches
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0
        val_batches = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                _, loss = model(batch_x, targets=batch_y)
                val_loss += loss.item()
                val_batches += 1
        avg_val_loss = val_loss / max(val_batches, 1)
        val_losses.append(avg_val_loss)

        elapsed = time.time() - start_time
        lr = scheduler.get_lr()

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_sft_model.pt')

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1}/{config.sft_epochs} | '
                  f'Train Loss: {avg_train_loss:.4f} | '
                  f'Val Loss: {avg_val_loss:.4f} | '
                  f'LR: {lr:.6f} | '
                  f'Time: {elapsed:.1f}s')

    print(f'\nSFT训练完成! 最佳验证Loss: {best_val_loss:.4f}')
    model.load_state_dict(torch.load('best_sft_model.pt', weights_only=True))
    return model, train_losses, val_losses

In [ ]:
print('开始SFT训练...')
model, sft_train_losses, sft_val_losses = train_sft(
    model, train_loader, val_loader, config, device
)

In [ ]:
print('SFT训练后测试推理效果:')
model.eval()
test_prompts = [
    '计算: 3 + 5 = ?',
    '计算: 9 - 4 = ?',
    '计算: 6 × 7 = ?',
    '计算: 8 ÷ 3 = ?',
    '计算: 2 - 7 = ?',
]
for prompt in test_prompts:
    input_ids = torch.tensor([[tokenizer.bos_id] + tokenizer.encode(prompt)], device=device)
    output_ids = model.generate(input_ids, max_new_tokens=80, temperature=0.3, top_k=20, eos_id=tokenizer.eos_id)
    output_text = tokenizer.decode_without_special(output_ids[0].tolist())
    print(f'输入: {prompt}')
    print(f'输出: {output_text}')
    print('---')

## Step 5: GRPO 强化学习

GRPO (Group Relative Policy Optimization) 是一种无需Critic网络的强化学习算法：
1. 对每个prompt采样一组响应
2. 基于答案正确性计算奖励
3. 在组内计算相对优势（advantage）
4. 使用裁剪目标函数更新策略
5. 加入KL散度惩罚防止偏离参考模型太远

In [ ]:
class GRPOTrainer:
    def __init__(self, policy_model, ref_model, tokenizer, config, device):
        self.policy = policy_model
        self.ref = ref_model
        self.tokenizer = tokenizer
        self.config = config
        self.device = device
        self.optimizer = torch.optim.AdamW(
            self.policy.parameters(),
            lr=config.grpo_lr,
            weight_decay=config.weight_decay,
            betas=(0.9, 0.95)
        )

    def compute_reward(self, generated_text, correct_answer):
        try:
            if '答案:' in generated_text:
                pred = generated_text.split('答案:')[-1].strip()
            elif '答案：' in generated_text:
                pred = generated_text.split('答案：')[-1].strip()
            else:
                return 0.0
            pred = pred.split('\n')[0].strip().rstrip('。')
            if pred == correct_answer:
                return 1.0
            if '余' in correct_answer and '余' in pred:
                parts_c = correct_answer.split('余')
                parts_p = pred.split('余')
                if len(parts_c) == 2 and len(parts_p) == 2:
                    if parts_c[0] == parts_p[0]:
                        return 0.5
            try:
                if pred.lstrip('-').isdigit() and correct_answer.lstrip('-').isdigit():
                    if abs(int(pred) - int(correct_answer)) <= 1:
                        return 0.3
            except:
                pass
            return 0.0
        except:
            return 0.0

    @torch.no_grad()
    def generate_group(self, prompt_text, group_size, max_new_tokens=80, temperature=0.8):
        self.policy.eval()
        prompt_ids = [self.tokenizer.bos_id] + self.tokenizer.encode(prompt_text)
        prompt_len = len(prompt_ids)
        prompt_tensor = torch.tensor([prompt_ids], device=self.device)

        all_full_ids = []
        all_gen_log_probs = []

        for _ in range(group_size):
            idx = prompt_tensor.clone()
            token_log_probs = []

            for _ in range(max_new_tokens):
                idx_cond = idx[:, -self.config.max_seq_len:]
                logits, _ = self.policy(idx_cond)
                raw_logits = logits[:, -1, :]
                raw_log_probs = F.log_softmax(raw_logits, dim=-1)
                sampled_logits = raw_logits / temperature
                probs = F.softmax(sampled_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                log_prob = raw_log_probs.gather(1, next_token).squeeze(-1)
                token_log_probs.append(log_prob)
                idx = torch.cat([idx, next_token], dim=1)
                if next_token.item() == self.tokenizer.eos_id:
                    break

            all_full_ids.append(idx[0].tolist())
            if token_log_probs:
                all_gen_log_probs.append(torch.stack(token_log_probs).sum())
            else:
                all_gen_log_probs.append(torch.tensor(0.0, device=self.device))

        return all_full_ids, all_gen_log_probs, prompt_len

    def compute_sequence_log_prob(self, model, input_ids_tensor, prompt_len):
        logits, _ = model(input_ids_tensor)
        log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
        targets = input_ids_tensor[:, 1:]
        token_log_probs = log_probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)
        gen_token_log_probs = token_log_probs[:, prompt_len - 1:]
        return gen_token_log_probs.sum()

    def train_step(self, prompts, answers):
        self.policy.train()
        self.ref.eval()
        group_size = self.config.grpo_group_size

        all_advantages = []
        all_full_ids = []
        all_old_log_probs = []
        all_prompt_lens = []

        for prompt, answer in zip(prompts, answers):
            full_ids_list, old_log_probs, prompt_len = self.generate_group(prompt, group_size)
            rewards = []
            for ids in full_ids_list:
                text = self.tokenizer.decode_without_special(ids)
                reward = self.compute_reward(text, answer)
                rewards.append(reward)

            rewards_tensor = torch.tensor(rewards, device=self.device)
            mean_r = rewards_tensor.mean()
            std_r = rewards_tensor.std() + 1e-8
            advantages = (rewards_tensor - mean_r) / std_r

            for j in range(group_size):
                all_full_ids.append(full_ids_list[j])
                all_advantages.append(advantages[j])
                all_old_log_probs.append(old_log_probs[j])
                all_prompt_lens.append(prompt_len)

        self.policy.train()
        self.optimizer.zero_grad()
        total_loss_val = 0.0
        n_valid = 0

        for i in range(len(all_full_ids)):
            ids = all_full_ids[i]
            if len(ids) < 2:
                continue
            input_tensor = torch.tensor([ids], device=self.device)
            prompt_len = all_prompt_lens[i]
            advantage = all_advantages[i]
            old_lp = all_old_log_probs[i].detach()

            new_lp = self.compute_sequence_log_prob(self.policy, input_tensor, prompt_len)
            with torch.no_grad():
                ref_lp = self.compute_sequence_log_prob(self.ref, input_tensor, prompt_len)

            log_ratio = torch.clamp(new_lp - old_lp, -5.0, 5.0)
            ratio = torch.exp(log_ratio)
            clipped_ratio = torch.clamp(
                ratio,
                1 - self.config.grpo_clip_ratio,
                1 + self.config.grpo_clip_ratio
            )

            policy_loss = -torch.min(ratio * advantage, clipped_ratio * advantage)
            kl_penalty = self.config.grpo_kl_coef * (new_lp - ref_lp)
            loss = (policy_loss + kl_penalty) / len(all_full_ids)
            loss.backward()
            total_loss_val += (policy_loss + kl_penalty).item()
            n_valid += 1

        if n_valid > 0:
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), self.config.grad_clip)
            self.optimizer.step()

        return total_loss_val / max(n_valid, 1)

    def train(self, train_data, config):
        print(f'开始GRPO训练, 共{config.grpo_steps}步...')
        prompts = [d['prompt'] for d in train_data]
        answers = [d['answer'] for d in train_data]

        scheduler = CosineScheduler(
            self.optimizer,
            warmup_steps=50,
            max_steps=config.grpo_steps,
            min_lr=1e-6
        )

        best_reward = 0.0
        step_losses = []
        start_time = time.time()

        for step in range(config.grpo_steps):
            indices = random.sample(range(len(prompts)), min(config.grpo_batch_size, len(prompts)))
            batch_prompts = [prompts[i] for i in indices]
            batch_answers = [answers[i] for i in indices]

            loss = self.train_step(batch_prompts, batch_answers)
            scheduler.step()
            step_losses.append(loss)

            if (step + 1) % 10 == 0:
                recent_loss = sum(step_losses[-10:]) / len(step_losses[-10:])
                lr = scheduler.get_lr()
                elapsed = time.time() - start_time
                print(f'Step {step+1}/{config.grpo_steps} | '
                      f'Loss: {recent_loss:.4f} | '
                      f'LR: {lr:.6f} | '
                      f'Time: {elapsed:.0f}s')

            if (step + 1) % 50 == 0:
                avg_reward = self.evaluate_reward(train_data[:50])
                print(f'  >> 评估: Avg Reward = {avg_reward:.3f}')
                if avg_reward > best_reward:
                    best_reward = avg_reward
                    torch.save(self.policy.state_dict(), 'best_grpo_model.pt')

        print(f'\nGRPO训练完成! 最佳平均奖励: {best_reward:.3f}')
        if os.path.exists('best_grpo_model.pt'):
            self.policy.load_state_dict(torch.load('best_grpo_model.pt', weights_only=True))
        return self.policy

    @torch.no_grad()
    def evaluate_reward(self, data):
        self.policy.eval()
        total_reward = 0
        for item in data:
            prompt_ids = torch.tensor(
                [[self.tokenizer.bos_id] + self.tokenizer.encode(item['prompt'])],
                device=self.device
            )
            output_ids = self.policy.generate(
                prompt_ids, max_new_tokens=80, temperature=0.1, top_k=10,
                eos_id=self.tokenizer.eos_id
            )
            output_text = self.tokenizer.decode_without_special(output_ids[0].tolist())
            total_reward += self.compute_reward(output_text, item['answer'])
        return total_reward / len(data)

In [ ]:
print('创建参考模型（SFT模型副本）...')
ref_model = GPTModel(config).to(device)
ref_model.load_state_dict(copy.deepcopy(model.state_dict()))
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False
print('参考模型已冻结。')

grpo_trainer = GRPOTrainer(model, ref_model, tokenizer, config, device)
model = grpo_trainer.train(train_data, config)

In [ ]:
print('GRPO训练后测试推理效果:')
model.eval()
test_prompts = [
    '计算: 3 + 5 = ?',
    '计算: 9 - 4 = ?',
    '计算: 6 × 7 = ?',
    '计算: 8 ÷ 3 = ?',
    '计算: 2 - 7 = ?',
    '计算: 9 ÷ 4 = ?',
    '计算: 0 × 8 = ?',
    '计算: 7 - 9 = ?',
]
for prompt in test_prompts:
    input_ids = torch.tensor([[tokenizer.bos_id] + tokenizer.encode(prompt)], device=device)
    output_ids = model.generate(input_ids, max_new_tokens=80, temperature=0.1, top_k=10, eos_id=tokenizer.eos_id)
    output_text = tokenizer.decode_without_special(output_ids[0].tolist())
    print(f'输入: {prompt}')
    print(f'输出: {output_text}')
    print('---')

## Step 6: 评估

在测试集上评估模型的推理准确率，按运算类型分别统计。

In [ ]:
@torch.no_grad()
def evaluate_model(model, test_data, tokenizer, device):
    model.eval()
    results = {'total': 0, 'correct': 0}
    op_results = {}

    for item in test_data:
        prompt = item['prompt']
        answer = item['answer']
        op = item['operation']

        input_ids = torch.tensor(
            [[tokenizer.bos_id] + tokenizer.encode(prompt)],
            device=device
        )
        output_ids = model.generate(
            input_ids, max_new_tokens=80, temperature=0.1, top_k=10,
            eos_id=tokenizer.eos_id
        )
        output_text = tokenizer.decode_without_special(output_ids[0].tolist())

        try:
            if '答案:' in output_text:
                pred = output_text.split('答案:')[-1].strip()
            elif '答案：' in output_text:
                pred = output_text.split('答案：')[-1].strip()
            else:
                pred = ''
            pred = pred.split('\n')[0].strip().rstrip('。')
        except:
            pred = ''

        is_correct = (pred == answer)
        results['total'] += 1
        if is_correct:
            results['correct'] += 1

        if op not in op_results:
            op_results[op] = {'total': 0, 'correct': 0}
        op_results[op]['total'] += 1
        if is_correct:
            op_results[op]['correct'] += 1

    overall_acc = results['correct'] / results['total'] * 100
    print(f'总体准确率: {results["correct"]}/{results["total"]} = {overall_acc:.1f}%')
    print()
    for op in sorted(op_results.keys()):
        r = op_results[op]
        acc = r['correct'] / r['total'] * 100
        op_name = {'+': '加法', '-': '减法', '×': '乘法', '÷': '除法'}.get(op, op)
        print(f'{op_name}({op}): {r["correct"]}/{r["total"]} = {acc:.1f}%')

    return overall_acc, op_results

print('=== SFT模型评估 ===')
sft_model = GPTModel(config).to(device)
sft_model.load_state_dict(torch.load('best_sft_model.pt', weights_only=True))
sft_acc, _ = evaluate_model(sft_model, test_data, tokenizer, device)

print(f'\n=== GRPO模型评估 ===')
grpo_acc, _ = evaluate_model(model, test_data, tokenizer, device)

print(f'\nGRPO相比SFT提升: {grpo_acc - sft_acc:+.1f}%')

## Step 7: 交互式推理演示

输入任意一位数算术题，模型将输出推理过程和答案。

In [ ]:
def interactive_inference(prompt, model, tokenizer, device, temperature=0.1, top_k=10):
    model.eval()
    input_ids = torch.tensor(
        [[tokenizer.bos_id] + tokenizer.encode(prompt)],
        device=device
    )
    output_ids = model.generate(
        input_ids, max_new_tokens=100, temperature=temperature, top_k=top_k,
        eos_id=tokenizer.eos_id
    )
    output_text = tokenizer.decode_without_special(output_ids[0].tolist())
    return output_text

demo_prompts = [
    '计算: 5 + 3 = ?',
    '计算: 9 - 2 = ?',
    '计算: 4 × 6 = ?',
    '计算: 7 ÷ 2 = ?',
    '计算: 0 + 0 = ?',
    '计算: 8 - 8 = ?',
    '计算: 9 × 9 = ?',
    '计算: 5 ÷ 3 = ?',
    '计算: 1 - 9 = ?',
    '计算: 3 × 0 = ?',
]

print('=' * 60)
print('从零构建AI推理大模型 - 推理演示')
print('=' * 60)
for prompt in demo_prompts:
    output = interactive_inference(prompt, model, tokenizer, device)
    print(f'\n输入: {prompt}')
    print(f'输出: {output}')
    print('-' * 40)

In [ ]:
print('保存最终模型...')
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {k: v for k, v in vars(config).items()},
    'tokenizer_char_to_id': tokenizer.char_to_id,
    'tokenizer_id_to_char': {int(k): v for k, v in tokenizer.id_to_char.items()},
}, 'final_model.pt')
print('模型已保存为 final_model.pt')
print('\n项目完成！')